# Schema Generalization QLoRA

Train a reduced-schema QLoRA model with schema-conditioned prompts, then evaluate on a mixed seen/unseen schema test set.

In [1]:
# Uncomment in a fresh Jupyter environment if needed.
# %pip install -q transformers datasets peft accelerate trl bitsandbytes pyyaml jsonschema

In [2]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'configs').exists() is False and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/home/lyan11/small-llm-structured-posttraining')

In [3]:
import torch

print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())

cuda_available = True
device = NVIDIA H200 NVL
bf16_supported = True


In [4]:
import yaml

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'train' / 'lora_phase1_qwen3b_reduced_h200fast.yaml'
assert CONFIG_PATH.exists(), f'Missing config: {CONFIG_PATH}'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config['experiment_name'] = 'qwen25_3b_schema_generalization_v1'
config['data']['train_file'] = 'data/processed/phase1_sft_train_reduced_schema_conditioned.jsonl'
config['data']['val_file'] = 'data/processed/phase1_sft_val_reduced_schema_conditioned.jsonl'
config['output']['save_dir'] = 'results/checkpoints/qwen25_3b_schema_generalization_v1'
config

{'experiment_name': 'qwen25_3b_schema_generalization_v1',
 'task_name': 'ticket_structured_output',
 'model': {'base_model': 'Qwen/Qwen2.5-3B-Instruct', 'max_seq_length': 2048},
 'training': {'method': 'qlora',
  'learning_rate': '2e-4',
  'num_train_epochs': 3,
  'per_device_train_batch_size': 16,
  'gradient_accumulation_steps': 2,
  'warmup_steps': 50},
 'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05},
 'data': {'train_file': 'data/processed/phase1_sft_train_reduced_schema_conditioned.jsonl',
  'val_file': 'data/processed/phase1_sft_val_reduced_schema_conditioned.jsonl'},
 'output': {'save_dir': 'results/checkpoints/qwen25_3b_schema_generalization_v1'}}

In [5]:
from datasets import load_dataset

train_file = PROJECT_ROOT / config['data']['train_file']
val_file = PROJECT_ROOT / config['data']['val_file']
assert train_file.exists(), f'Missing train file: {train_file}'
assert val_file.exists(), f'Missing val file: {val_file}'

dataset = load_dataset(
    'json',
    data_files={
        'train': str(train_file),
        'validation': str(val_file),
    },
)
dataset

DatasetDict({
    train: Dataset({
        features: ['sample_id', 'messages', 'metadata'],
        num_rows: 1993
    })
    validation: Dataset({
        features: ['sample_id', 'messages', 'metadata'],
        num_rows: 247
    })
})

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

model_name = config['model']['base_model']
max_seq_length = int(config['model']['max_seq_length'])
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_chat_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    example['text'] = text
    return example

dataset = dataset.map(format_chat_example)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=int(config['lora']['r']),
    lora_alpha=int(config['lora']['alpha']),
    lora_dropout=float(config['lora']['dropout']),
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

Map:   0%|          | 0/1993 [00:00<?, ? examples/s]

Map:   0%|          | 0/247 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [7]:
training_args = TrainingArguments(
    output_dir=config['output']['save_dir'],
    learning_rate=float(config['training']['learning_rate']),
    num_train_epochs=int(config['training']['num_train_epochs']),
    per_device_train_batch_size=int(config['training']['per_device_train_batch_size']),
    per_device_eval_batch_size=int(config['training']['per_device_train_batch_size']),
    gradient_accumulation_steps=int(config['training']['gradient_accumulation_steps']),
    warmup_steps=int(config['training']['warmup_steps']),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer

Tokenizing train dataset:   0%|          | 0/1993 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1993 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/247 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/247 [00:00<?, ? examples/s]

In [8]:
train_result = trainer.train()
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.366273,0.344627
100,0.273898,0.268593
150,0.229790,0.238923


TrainOutput(global_step=189, training_loss=0.47074108527450964, metrics={'train_runtime': 216.9312, 'train_samples_per_second': 27.562, 'train_steps_per_second': 0.871, 'total_flos': 5.668287706580582e+16, 'train_loss': 0.47074108527450964})

In [9]:
output_dir = PROJECT_ROOT / config['output']['save_dir']
output_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))
print('saved to', output_dir)
print('exists =', output_dir.exists())
print([p.name for p in output_dir.glob('*')][:20])

saved to /home/lyan11/small-llm-structured-posttraining/results/checkpoints/qwen25_3b_schema_generalization_v1
exists = True
['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


## Seen / Unseen Inference Export

In [13]:
from src.data.io import load_jsonl
from pathlib import Path

test_path = PROJECT_ROOT / "data" / "generalization" / "phase1_test_seen_unseen_reduced.jsonl"
test_records = load_jsonl(test_path)

print(len(test_records))
print(test_records[0]["sample_id"])
print(test_records[255]["sample_id"])
print(test_records[-1]["sample_id"])


508
console_ai_it_helpdesk_tickets-000409-sai67ttf1::seen
console_ai_it_helpdesk_tickets-000017-xka1oq83l::unseen
console_ai_it_helpdesk_tickets-000192-gzm0m2o5i::unseen


In [14]:
from pathlib import Path

prediction_path = PROJECT_ROOT / "results" / "predictions" / "qwen25_3b_schema_generalization_v1_test.jsonl"
if prediction_path.exists():
    prediction_path.unlink()
    print("deleted old prediction file")
else:
    print("no old prediction file")


deleted old prediction file


In [15]:
from src.data.io import load_jsonl, dump_jsonl
from src.evaluation.metrics import try_parse_prediction_text
from src.schemas.registry import get_schema

test_path = PROJECT_ROOT / 'data' / 'generalization' / 'phase1_test_seen_unseen_reduced.jsonl'
prediction_path = PROJECT_ROOT / 'results' / 'predictions' / 'qwen25_3b_schema_generalization_v1_test.jsonl'
prediction_path.parent.mkdir(parents=True, exist_ok=True)

assert test_path.exists(), f'Missing test file: {test_path}'
test_records = load_jsonl(test_path)
len(test_records)

508

In [16]:
model.eval()
predictions = []

for idx, record in enumerate(test_records):
    schema_definition = json.dumps(get_schema(record['schema_name']), ensure_ascii=False, separators=(',', ':'))
    user_prompt = (
        f"Task: extract a structured record for {record['task_name']}.\n"
        f"Schema name: {record['schema_name']}\n"
        f"Schema definition:\n{schema_definition}\n"
        'Return a JSON object only.\n'
        'Input text:\n'
        f"{record['input_text']}"
    )
    messages = [
        {'role': 'system', 'content': 'You are an information extraction model. Return only JSON that matches the requested schema. Do not add explanations or markdown.'},
        {'role': 'user', 'content': user_prompt},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=192,
            do_sample=False,
            temperature=None,
            top_p=None,
            use_cache=True,
        )
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    _, parsed = try_parse_prediction_text(generated)
    predictions.append({
        'sample_id': record['sample_id'],
        'prediction_text': generated,
        'prediction_json': parsed,
        'metadata': {
            'model_name': model_name,
            'experiment_id': config['experiment_name'],
            'schema_seen_status': record.get('schema_seen_status'),
            'schema_name': record['schema_name'],
        },
    })
    if idx % 50 == 0:
        print('processed', idx)

dump_jsonl(prediction_path, predictions)
print('saved predictions to', prediction_path)
print('exists =', prediction_path.exists())

processed 0
processed 50
processed 100
processed 150
processed 200
processed 250
processed 300
processed 350
processed 400
processed 450
processed 500
saved predictions to /home/lyan11/small-llm-structured-posttraining/results/predictions/qwen25_3b_schema_generalization_v1_test.jsonl
exists = True


In [12]:
for item in predictions[:2]:
    print(item)
    print('---')

{'sample_id': 'console_ai_it_helpdesk_tickets-000409-sai67ttf1', 'prediction_text': '{"summary": "Password Reset Assistance Needed", "category": "task", "priority": "medium", "requires_followup": true, "affected_systems": [{"name": "Account", "component": "account"}], "actions_requested": [{"action": "Handle request: Password Reset Assistance Needed", "owner": null, "deadline": null}], "constraints": {"environment": null, "blocking": null}}', 'prediction_json': {'summary': 'Password Reset Assistance Needed', 'category': 'task', 'priority': 'medium', 'requires_followup': True, 'affected_systems': [{'name': 'Account', 'component': 'account'}], 'actions_requested': [{'action': 'Handle request: Password Reset Assistance Needed', 'owner': None, 'deadline': None}], 'constraints': {'environment': None, 'blocking': None}}, 'metadata': {'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'experiment_id': 'qwen25_3b_schema_generalization_v1', 'schema_seen_status': 'seen', 'schema_name': 'ticket_schema_v1_